# 🧩 Mini-Lab: Files and Paths That Work on Anyone's Machine

**Module: Real Data: Paths, CSV, JSON, pandas, NumPy** | **Type: Mini-Lab (Brick)**

---

## Learning Objectives

By the end of this mini-lab, you will be able to:

1. **Understand** why hardcoded string paths break across operating systems
2. **Use** `pathlib.Path` objects to build portable, OS-agnostic file paths
3. **Declare** text encodings explicitly so files read identically on every machine

## Target Concepts

| Concept | Gloss |
|---------|-------|
| Filesystem paths | Use Path objects, not strings, to build file locations that work across OSes. Prefer relative paths anchored to your project (not your shell), since relative paths depend on the current working directory. |
| Text encoding | Bytes become text only under a declared encoding, usually UTF-8. If the sender and reader assume different encodings, you get errors or corrupted characters, even when the file "looks" like plain text. |

## Why This Matters

Two of the most common "works on my machine" bugs in data projects:

1. **String paths with `/` or `\`** — a path that works on Mac silently breaks on Windows (and vice versa).
2. **Missing encoding declarations** — a file with accented characters (é, ñ, ü) opens fine on one OS but produces garbled text or crashes on another, because the default encoding differs.

`pathlib.Path` and explicit `encoding="utf-8"` eliminate both problems in one line each.

In [1]:
from pathlib import Path
from IPython.display import display, Markdown

def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

## The Problem with String Paths

> 🧩 **Filesystem paths.** Recall the video's Path-objects-over-strings demo; here it is in action.

Let's see why `"data/files/report.txt"` is fragile.

In [2]:
import os

# String concatenation — the WRONG way
bad_path = "data" + "/" + "files" + "/" + "report.txt"
print(f"Hardcoded string path: {bad_path}")
print(f"OS path separator:     {os.sep}")
print()

# pathlib — the RIGHT way
good_path = Path("data") / "files" / "report.txt"
print(f"pathlib Path object:   {good_path}")
print(f"Type:                  {type(good_path).__name__}")
print()
print("The / operator builds the correct separator for YOUR OS automatically.")

Hardcoded string path: data/files/report.txt
OS path separator:     \

pathlib Path object:   data\files\report.txt
Type:                  WindowsPath

The / operator builds the correct separator for YOUR OS automatically.


## Essential `Path` Operations

A `Path` object gives you everything you need without string gymnastics.

In [3]:
p = Path("data") / "reviews" / "café_ratings.csv"

md(
    f"| Property | Value |\n"
    f"|----------|-------|\n"
    f"| `p.name` (filename) | `{p.name}` |\n"
    f"| `p.stem` (without extension) | `{p.stem}` |\n"
    f"| `p.suffix` (extension) | `{p.suffix}` |\n"
    f"| `p.parent` (directory) | `{p.parent}` |\n"
    f"| `p.parts` (all pieces) | `{p.parts}` |\n"
    f"| `p.is_absolute()` | `{p.is_absolute()}` |\n"
)

| Property | Value |
|----------|-------|
| `p.name` (filename) | `café_ratings.csv` |
| `p.stem` (without extension) | `café_ratings` |
| `p.suffix` (extension) | `.csv` |
| `p.parent` (directory) | `data\reviews` |
| `p.parts` (all pieces) | `('data', 'reviews', 'café_ratings.csv')` |
| `p.is_absolute()` | `False` |


## Creating Directories Safely

`Path.mkdir()` with `parents=True` and `exist_ok=True` is the safe, cross-platform way to ensure a folder exists before writing to it.

In [4]:
import tempfile

# Use a temp directory so this lab leaves no mess
tmp = Path(tempfile.mkdtemp())
output_dir = tmp / "output" / "cleaned"

# Create nested dirs in one call — works on Mac, Windows, Linux
output_dir.mkdir(parents=True, exist_ok=True)

print(f"Created: {output_dir}")
print(f"Exists:  {output_dir.exists()}")

Created: C:\Users\molak\AppData\Local\Temp\tmp4ww3pabt\output\cleaned
Exists:  True


## The Encoding Trap

> 🧩 **Text encoding.** Declare `encoding="utf-8"` so files read the same on every machine.

Different operating systems use different **default** encodings:

| OS | Typical Default |
|----|----------------|
| macOS / Linux | `utf-8` |
| Windows (older) | `cp1252` (Western European) |

If you omit `encoding=`, Python uses the OS default. A file written on Mac may produce garbled text — or crash — on Windows.

**The fix is simple: always declare `encoding="utf-8"`.**

In [5]:
# Text with non-ASCII characters (accents, emoji, etc.)
sample_text = "Café résumé — naïve señor 🎯\nLine two: über cool"

file_path = output_dir / "sample.txt"

# ✅ GOOD: explicit encoding
file_path.write_text(sample_text, encoding="utf-8")
print(f"Wrote {file_path.stat().st_size} bytes to {file_path.name}")

Wrote 59 bytes to sample.txt


In [6]:
# ✅ GOOD: read with the same explicit encoding
content = file_path.read_text(encoding="utf-8")

# Verify round-trip
assert content == sample_text, "Round-trip failed!"
print("✅ Round-trip successful — text is identical after write → read")

✅ Round-trip successful — text is identical after write → read


## What Goes Wrong Without `encoding=`

Let's simulate the problem by writing in one encoding and reading in another.

In [7]:
# Write as utf-8
mismatch_path = output_dir / "mismatch.txt"
mismatch_path.write_text("Café résumé", encoding="utf-8")

# Read as latin-1 (simulates a Windows default mismatch)
wrong_read = mismatch_path.read_text(encoding="latin-1")

print(f"Original:          Café résumé")
print(f"Mismatched read:   {wrong_read}")
print()
print("☝️ The accented characters are garbled — this is called 'mojibake'.")
print("   Always declare encoding='utf-8' on BOTH write and read.")

Original:          Café résumé
Mismatched read:   CafÃ© rÃ©sumÃ©

☝️ The accented characters are garbled — this is called 'mojibake'.
   Always declare encoding='utf-8' on BOTH write and read.


## Putting It Together: A Safe Loader Function

Here's a minimal pattern you can reuse in any project — portable path + declared encoding.

In [8]:
def safe_load(filepath: str | Path, encoding: str = "utf-8") -> str:
    """Load a text file using a Path object and explicit encoding."""
    p = Path(filepath)
    if not p.exists():
        raise FileNotFoundError(f"File not found: {p.resolve()}")
    return p.read_text(encoding=encoding)


def safe_save(filepath: str | Path, content: str, encoding: str = "utf-8") -> Path:
    """Save text to a file, creating parent dirs if needed."""
    p = Path(filepath)
    p.parent.mkdir(parents=True, exist_ok=True)
    p.write_text(content, encoding=encoding)
    return p


# Demo
demo_path = output_dir / "demo" / "greeting.txt"
safe_save(demo_path, "Héllo, wörld! 🌍")
result = safe_load(demo_path)
print(f"Loaded: {result}")
print(f"Path:   {demo_path}")

Loaded: Héllo, wörld! 🌍
Path:   C:\Users\molak\AppData\Local\Temp\tmp4ww3pabt\output\cleaned\demo\greeting.txt


In [9]:
# Clean up temp directory
import shutil
shutil.rmtree(tmp)
print(f"Cleaned up temp dir: {tmp}")

Cleaned up temp dir: C:\Users\molak\AppData\Local\Temp\tmp4ww3pabt

## 🎯 Summary

### Key Takeaways

1. **`pathlib.Path` over string paths** — the `/` operator builds OS-correct separators so your code runs on Mac, Windows, and Linux without changes
2. **Always declare `encoding="utf-8"`** — omitting it lets Python fall back to the OS default, which causes garbled text (mojibake) when files move between machines
3. **`mkdir(parents=True, exist_ok=True)`** — safely creates nested directories without crashing if they already exist
4. **Two-line safe pattern** — `Path` for the path + `encoding="utf-8"` for the read/write eliminates the two most common cross-platform file bugs